# 01 - LLM Foundations

This notebook builds the mental model needed before using LLMs in applications. The goal is not to derive transformer math. The goal is to understand the moving parts well enough to reason about prompts, context windows, retrieval, cost, and failure modes.


## Why The Generation Loop Matters

LLM applications combine model behavior with software design choices. The model predicts text, while the application decides which instructions, tools, retrieved sources, decoding settings, and validation rules surround that prediction. This distinction matters because many production issues are system issues rather than model issues: missing context, weak retrieval, permissive prompts, or absent checks.

Read this notebook as the architecture layer for the rest of the repository. The later notebooks use tokens, embeddings, semantic search, and evaluation to make that architecture more concrete.

**An LLM call is a packaged request.**

An LLM application is easier to reason about when the model call is treated as a packaged request rather than a mysterious conversation. The request contains instructions, user input, optional examples, optional retrieved context, and generation settings. The model receives all of that as tokens. It then produces a probability distribution for the next token, selects one token according to the decoding strategy, appends it to the visible context, and repeats the process until it reaches a stop condition.

This means the application controls more than the model alone. It controls what evidence is visible, how much answer space is reserved, how deterministic the sampling should be, and whether unsupported answers are allowed. A strong LLM system therefore combines model behavior with software controls: retrieval, validation, fallbacks, monitoring, and clear product rules.


**Engineering lens.**

When debugging LLM behavior, separate three layers: input construction, model generation, and output handling. Input construction decides what the model can see. Generation decides how text is sampled from probabilities. Output handling decides whether the answer is accepted, parsed, cited, retried, or rejected. Many problems that look like model failures are actually application design failures: missing context, conflicting instructions, no fallback path, or an output format that was never validated.


## Objectives

After completing this notebook, you should be able to:

- explain why LLMs are next-token prediction systems;
- distinguish base, instruction-tuned, and chat models;
- trace the path from raw prompt to generated text;
- explain why context, decoding, and retrieval choices affect reliability.


## Workflow

```mermaid
flowchart TD
    A[Raw text] --> B[Tokens]
    B --> C[Token embeddings]
    C --> D[Transformer context]
    D --> E[Logits]
    E --> F[Decoding]
    F --> G[Generated token]
    G --> B
```


## Mental Model: Prediction, Not Truth Lookup

A useful first habit is to separate **language generation** from **fact lookup**. During generation, the model chooses likely next tokens based on its parameters and the visible context. That context can include retrieved documents, tool results, system instructions, examples, and the user question, but the core operation is still token prediction.

This matters in applications because the model may produce an answer that is fluent, well formatted, and still unsupported. A production system usually needs additional grounding and checks around the model: retrieval for source material, tools for current facts, validation for output structure, and fallback behavior when the evidence is weak.


## What Is An LLM?

An LLM is a neural network trained on large collections of text. During pretraining, the model learns statistical patterns that help it predict missing or next tokens. Many production models are then instruction-tuned so they behave more like helpful assistants.


**Before running the next cell:** compare model families by interface and behavior, not by brand name.

The next cell uses a small table-like list of dictionaries. Each dictionary names a model family, describes the behavior that matters for applications, and shows the prompt shape that usually goes with that family. This is not meant to simulate a real provider API. It gives you a compact way to compare model categories by interface and behavior before you choose a model for a product workflow.


In [ ]:
# Compare model families by behavior instead of treating every LLM as the same product.
model_types = [
    {
        "type": "base model",
        "behavior": "continues text",
        "example_prompt": "The capital of France is",
    },
    {
        "type": "instruction-tuned model",
        "behavior": "follows task instructions",
        "example_prompt": "Explain Paris in one sentence.",
    },
    {
        "type": "chat model",
        "behavior": "uses role-formatted messages",
        "example_prompt": "system + user + assistant messages",
    },
]
for row in model_types:
    print(f"{row['type']}: {row['behavior']} | prompt shape: {row['example_prompt']}")

## The Core Prediction Loop

LLM generation is iterative. The model does not write a full answer in one step. It predicts a distribution over the next token, chooses one token through a decoding strategy, appends that token to the context, and repeats.


**Before running the next cell:** follow the loop as context grows one selected token at a time.

The generation example keeps a list of preselected tokens so the loop is visible. A real model would compute probabilities from neural network weights at every step. Here, the important idea is sequencing: each selected token becomes part of the next context. That is why early tokens can steer later tokens, why long outputs can drift, and why stop conditions matter.


In [ ]:
# Simulate generation one token at a time to make the prediction loop visible.
prompt_tokens = ["Explain", "token", "cost"]
generated_tokens = []
for next_token in ["in", "one", "sentence", "."]:
    context = prompt_tokens + generated_tokens
    print("context ->", " ".join(context))
    print("selected next token ->", next_token)
    generated_tokens.append(next_token)
print("final text ->", " ".join(prompt_tokens + generated_tokens))

## Transformer Flow In Application Terms

```mermaid
flowchart TD
    A[Application builds prompt] --> B[Tokenizer creates token IDs]
    B --> C[Embedding lookup]
    C --> D[Transformer blocks update context]
    D --> E[Logits over vocabulary]
    E --> F[Decoder selects next token]
    F --> G{Stop condition met?}
    G -- No --> D
    G -- Yes --> H[Application receives final text]
```

You do not need to implement this flow by hand, but you do need to know where application design choices enter the system. Prompt structure changes the input tokens. Retrieval changes the context. Decoding settings change how deterministic the output is. Stop conditions and output schemas change whether the application can reliably parse the result.


## Important Vocabulary

- **Token:** a unit of text used by the model. It can be a word, subword, punctuation mark, or control symbol.
- **Embedding:** a vector representation of a token, sentence, or document.
- **Context window:** the maximum number of tokens the model can inspect at once.
- **Logits:** raw scores for possible next tokens before probabilities.
- **Decoding:** the process of choosing the next token from model scores.


## Why Transformers Matter

Transformers use attention so tokens can use information from other tokens in the context. In the sentence `The analyst uploaded the notebook because it contains the exercise`, attention helps connect `it` to `the notebook`.


In [ ]:
# Use a tiny attention-style example to show how tokens can depend on earlier context.
sentence = "The analyst uploaded the notebook because it contains the exercise."
references = {
    "it": "the notebook",
    "because": "reason connector",
    "exercise": "content clue",
}
print(sentence)
for token, role in references.items():
    print(f"{token!r}: {role}")

## Decoding Settings

Greedy decoding, temperature, top-k, and top-p control how deterministic or varied generated text can be. Lower randomness is useful for structured tasks. More randomness can help brainstorming but increases risk.


## Determinism Versus Diversity

Decoding settings are operational controls. Greedy decoding chooses the most likely token and is useful when reproducibility matters. Temperature changes how sharply or broadly probabilities are sampled. Top-k and top-p restrict the candidate set before sampling. In product terms, these settings decide whether the system should behave like a stable extractor, a drafting assistant, or a brainstorming partner.

The setting is not good or bad by itself. It must match the workflow. A compliance classifier, JSON extractor, or routing decision should usually be deterministic. A creative naming task or exploratory draft can tolerate more variation. The same model can feel reliable or chaotic depending on these controls.


In [ ]:
# Summarize decoding settings as operational choices for application behavior.
settings = [
    {"setting": "greedy", "determinism": "high", "best_for": "structured extraction"},
    {
        "setting": "low temperature",
        "determinism": "high",
        "best_for": "tests and reproducible demos",
    },
    {
        "setting": "higher temperature",
        "determinism": "lower",
        "best_for": "brainstorming alternatives",
    },
    {"setting": "top-p", "determinism": "medium", "best_for": "bounded sampling"},
]
for row in settings:
    print(
        f"{row['setting']}: determinism={row['determinism']}, best for {row['best_for']}"
    )

## Why LLMs Are Not Databases

LLMs can generate plausible answers, but they are not reliable databases. Use external sources for authoritative facts, current business records, and auditable information.


## Common Misunderstandings

| Misunderstanding | Better framing |
| --- | --- |
| The model answers from a database. | The model generates likely text from parameters and context. Use databases or retrieval for authoritative facts. |
| A longer prompt is always better. | More context can help, but irrelevant context can distract the model and increase cost. |
| Temperature makes the model smarter. | Temperature changes sampling behavior. It does not add knowledge or verification. |
| Retrieval guarantees correctness. | Retrieval improves grounding only if the retrieved sources are relevant, current, and trusted. |

When debugging LLM behavior, avoid asking only “is the model good?” Ask which part of the system produced the failure: prompt, retrieval, model, decoding, output validation, or monitoring.


```mermaid
flowchart TD
    U[User question] --> R[Retrieve sources]
    R --> P[Build grounded prompt]
    P --> L[LLM response]
    L --> C[Check and cite]
    C --> A[Answer]
```


## What To Inspect

Check whether you can explain each step in the generation loop without saying “the model just knows.” If you cannot name where context, retrieval, or decoding enters the workflow, revisit the diagram.


## Checkpoint

Answer the questions, then compare with the reference answers below:

1. Why can an LLM be fluent and wrong?
2. What goes into a context window?
3. When should an application use retrieval or an API instead of relying on model memory?
4. Which decoding setting would you choose for a JSON extraction task, and why?


## Checkpoint Reference Answers

1. An LLM can be fluent and wrong because fluency comes from likely token generation, not from verified evidence.
2. A context window contains visible input: system instructions, examples, conversation history, retrieved documents, tool results, the user request, and generated output tokens.
3. Use retrieval or an API when the answer depends on current, private, auditable, or authoritative information.
4. Use greedy decoding or a very low temperature for JSON extraction because deterministic output is easier to validate and reproduce.